# Lab 1: Agents as Tools for an expense assistant

An employee asks whether an expense can be submitted as it is. Answering takes three kinds of records: the receipt, the company expense policy, and the employee's prior claims. In this lab an orchestrator Agent answers the question by calling three specialist Agents as tools. Each specialist owns one kind of record.

The orchestrator model decides which specialists a request needs, writes the input each one receives, and combines their answers. The records are not in this notebook. They sit behind an Amazon Bedrock AgentCore Gateway, which serves an AWS Lambda function as Model Context Protocol (MCP) tools, so every fact in the answer has to come back through a specialist's MCP tool call.

## What you will do

1. Run the orchestrator and watch each Agent in its own pane.
2. Read the evidence: every specialist call with its input and result, and the MCP tool calls behind them.
3. Watch a fact get lost between two specialists, then see a small result contract carry it across.

**Estimated time:** 20-25 minutes.

You do not change the agent code. Run the cells from the top in order. If you plan to complete optional Lab 6, run the clearly marked observability cell before the first Gateway call; otherwise skip that cell.

![The expense coordinator calls three specialists as tools, choosing which of them run. Each specialist reads one kind of record through AgentCore Gateway, which serves an expense Lambda function over MCP.](images/topology.png)


## Why this is Agents as Tools

Requests differ in what they need. "What is the total on this receipt?" needs one specialist, and "Can this be submitted as it is?" needs all three, in an order set by which facts each one depends on. A model is a reasonable judge of that, so the orchestrator model owns the route, and it stays responsible for the answer the employee sees.

In Strands, an Agent passed in another Agent's `tools` list becomes a tool. Its `name` is the tool name and its `description` is what the orchestrator model reads when it decides whether to call it. The tool takes one string, `input`. That string is the whole handoff: a specialist knows its own prompt, its own tools, and the input the orchestrator wrote, nothing else.

Every specialist call is another model call, and the orchestrator takes one more turn to read each result. When the steps are the same for every request, a fixed route in your own code is cheaper and easier to test.

## Setup

Run the `%pip` cell by itself first. If it installs or upgrades anything, restart the kernel. Then run the import, model, and lab-resource cells in order.

When you reach **Optional: Enable observability**, run that cell if you plan to complete Lab 6; otherwise skip it. Then select the **List the Gateway tools** cell and choose **Run > Run All Below**. Starting **Run All** above the install cell can stop after a package-triggered kernel restart.

In [ ]:
%pip install -r requirements.txt

In [ ]:
import json
import sys
from pathlib import Path
from typing import Literal

import boto3
import httpx
from pydantic import BaseModel, Field
from strands import Agent
from strands.models import BedrockModel
from strands.tools.mcp import MCPClient

sys.path.insert(0, str(Path("..").resolve()))
from utils.live_panes import LivePanes
from utils.stacks import cleanup, find_or_deploy
from utils.validation import write_validation_artifact

The specialists run on Amazon Nova Pro. Each does one lookup and a short report. The orchestrator runs on Anthropic Claude Sonnet 5, because it plans the route and writes every handoff. Both models run in the AWS Region of this notebook's session.

In [ ]:
region = boto3.Session().region_name
nova = BedrockModel(model_id="us.amazon.nova-pro-v1:0", region_name=region)
sonnet = BedrockModel(model_id="global.anthropic.claude-sonnet-5", region_name=region)

### Lab resources

The lab stack holds the Lambda function that answers three read-only tools, `get_receipt`, `get_expense_policy`, and `list_prior_claims`, from fixed workshop data. It also holds the AgentCore Gateway that serves the function as an MCP target named `expenses`, and an Amazon Cognito user pool whose access tokens are the only ones the Gateway accepts. At an event the stack already exists. On your own account, the next cell deploys it, which takes about two minutes.

The cell then gets an access token with the OAuth client-credentials grant. The client secret is not a stack output, so the cell reads it from Amazon Cognito.

The optional observability cell comes next, before the first Gateway request. Run it only if you plan to complete Lab 6.

In [ ]:
# Find the lab stack, or deploy it in your own account.
outputs = find_or_deploy("agents-as-tools", "infra/agents-as-tools.yaml")

# Get a one-hour Gateway access token from Amazon Cognito.
cognito = boto3.client("cognito-idp", region_name=region)
# Cognito generated the app client's secret, so read it from the user pool client.
client_secret = cognito.describe_user_pool_client(
    UserPoolId=outputs["UserPoolId"],
    ClientId=outputs["ClientId"],
)["UserPoolClient"]["ClientSecret"]
# The OAuth client credentials grant: the client ID and secret go in a Basic auth header.
token = httpx.post(
    outputs["TokenUrl"],
    data={
        "grant_type": "client_credentials",
        "scope": outputs["Scope"],
    },
    auth=(outputs["ClientId"], client_secret),
)
token.raise_for_status()

# The Gateway URL, auth header, and target name for the MCP clients.
GATEWAY_URL = outputs["GatewayUrl"]
GATEWAY_HEADERS = {"Authorization": f"Bearer {token.json()['access_token']}"}  # sent on every MCP request
TARGET = outputs["TargetName"]

print("Gateway ready     :", GATEWAY_URL)
print("Gateway target    :", TARGET)

### Optional: Enable observability

Run the next cell if you plan to complete **Lab 6: Observability**. It enables Transaction Search, 100% trace indexing, Gateway application logs, and managed Gateway traces before Lab 1 makes its first Gateway request.

Continue only after the cell prints `Gateway traces : enabled`. The configuration and its cleanup ownership are saved for Lab 6, so the Gateway activity you generate in this lab remains available there. If you are not completing Lab 6, skip this cell.

In [ ]:
from utils.observability import enable_gateway_observability

observability_state_path = Path("..") / ".lab6-observability-state.json"
lab6_observability = enable_gateway_observability(
    outputs,
    observability_state_path,
    region=region,
)
assert lab6_observability["trace_delivery_ready"], (
    "Gateway trace delivery is still activating. Wait a few minutes and rerun this cell before continuing."
)

### List the Gateway tools

The next cell makes the first Gateway request by listing the whole tool catalog once, without a filter.

**What to look for:** three tools, each prefixed with the target name, for example `expenses___get_receipt`. The token is valid for one hour. If a later cell fails with `401 Unauthorized`, rerun the lab resources cell and then the cell that failed.

In [ ]:
# The with block opens the MCP session and closes it at the end. Gateway tool names are <target>___<tool>.
with MCPClient(url=GATEWAY_URL, headers=GATEWAY_HEADERS) as catalog:
    print("Gateway tools:", sorted(tool.tool_name for tool in catalog.list_tools_sync()))

## The tools

Each specialist gets its own `MCPClient` with a `tool_filters` allow list, so each one is shown only its own Gateway tool. The orchestrator gets no Gateway tools at all. Its only tools are the specialists.

An `MCPClient` in an Agent's `tools` list is a tool provider: Strands opens the connection and lists the tools when the Agent is created, and closes it when no Agent uses the client any more. Several Agents can share one client, so the cells below define new Agents but reuse these three clients.

In [ ]:
# One MCP client per specialist, all on the same Gateway. "allowed" keeps only the tool with that exact name.
# No with block here: an Agent opens the client's session when it loads the client's tools.
receipt_tools = MCPClient(
    url=GATEWAY_URL,
    headers=GATEWAY_HEADERS,
    tool_filters={"allowed": [f"{TARGET}___get_receipt"]},
)
policy_tools = MCPClient(
    url=GATEWAY_URL,
    headers=GATEWAY_HEADERS,
    tool_filters={"allowed": [f"{TARGET}___get_expense_policy"]},
)
audit_tools = MCPClient(
    url=GATEWAY_URL,
    headers=GATEWAY_HEADERS,
    tool_filters={"allowed": [f"{TARGET}___list_prior_claims"]},
)

## What to look for

The request asks whether meal expense `expense-1042` can be submitted as it is. That depends on the meals policy and on whether the employee claimed the same meal before. Only the receipt holds the total and the employee ID the other two specialists need.

Look for:

- Which specialists does the orchestrator call, and in how many turns?
- Which specialists run in the same turn?
- Which facts from the receipt appear in the policy and audit inputs?


## Run the orchestrator

The cell defines the three specialists and the orchestrator, then streams the request with `orchestrator.stream_async(request)`. Watch which specialists the orchestrator selects, the order it calls them, and which evidence tool each specialist uses.

Every Agent sets `callback_handler=None`: `stream_async` still calls each Agent's default handler, which would print the same text a second time below the panes.

**What to look for:** the orchestrator calls `receipt_agent` before `policy_agent` and `audit_agent`, and each specialist makes one `expenses___...` tool call.

In [ ]:
# The specialists. Each description is what the orchestrator reads when it chooses whom to call.
receipt_agent = Agent(
    name="receipt_agent",
    description="Looks up the receipt of one expense by expense ID and reports what is on it.",
    model=nova,
    system_prompt=(
        "You are an expense receipt specialist. Look up the receipt with your tool and report only facts that "
        "appear in it. Do not interpret policy. Answer in at most four sentences."
    ),
    tools=[receipt_tools],  # the filtered MCP client: one Gateway tool
    callback_handler=None,  # no printing to the cell; LivePanes shows the stream instead
)

policy_agent = Agent(
    name="policy_agent",
    description=(
        "Looks up the company expense policy for a category and, given the expense facts in its input, says "
        "whether an exception is required and who must approve it."
    ),
    model=nova,
    system_prompt=(
        "You are an expense policy specialist. Look up the policy for the expense's category with your tool and "
        "apply only that policy to the facts in the request. If a fact you need is missing from the request, say "
        "which one. Answer in at most four sentences."
    ),
    tools=[policy_tools],
    callback_handler=None,
)

audit_agent = Agent(
    name="audit_agent",
    description=(
        "Looks up an employee's prior expense claims by employee ID and, given the expense facts in its input, "
        "says whether the claim appears to duplicate a prior claim."
    ),
    model=nova,
    system_prompt=(
        "You are an expense audit specialist. Look up the employee's prior claims with your tool, compare them "
        "with the facts in the request, and say whether the claim appears to duplicate one of them. If a fact you "
        "need is missing from the request, say which one. Answer in at most four sentences."
    ),
    tools=[audit_tools],
    callback_handler=None,
)

# The orchestrator. Its prompt says how to delegate and names no specialist.
COORDINATOR_PROMPT = (
    "You coordinate employee expense questions. You cannot see receipts, policies, or prior claims yourself; "
    "the specialists hold them. Call only the specialists the request needs. Each specialist sees only the input "
    "you send it, so when one specialist needs facts that another holds, get those facts first and include them "
    "in your input. Do not invent facts or rules. Answer in at most four sentences."
)
orchestrator = Agent(
    name="expense_coordinator",
    model=sonnet,
    system_prompt=COORDINATOR_PROMPT,
    tools=[receipt_agent, policy_agent, audit_agent],  # each Agent becomes a tool with one `input` string
    callback_handler=None,
)

# The request.
request = (
    "Can the meal expense expense-1042 be submitted as it is? Check whether it needs a policy exception, "
    "and whether it duplicates a prior claim."
)

# Run and stream each agent into its pane.
# A specialist's events arrive inside the orchestrator's tool_stream events; LivePanes gives each its own pane.
panes = LivePanes()
async for event in orchestrator.stream_async(request):
    panes.show(event)

## Inspect the evidence

The orchestrator's message history holds every specialist call: the turn it was made in, the input the orchestrator wrote, and the result that came back. Each specialist's own history holds the MCP tool calls it made through the Gateway. By default Strands resets a specialist's history before each call, so what you read here is its most recent call.

Compare this output with the questions under **What to look for**.

In [ ]:
# Group the specialist calls by orchestrator turn.
# In the history, an assistant message holds the toolUse blocks of one turn,
# and the next user message holds their toolResult blocks, matched by toolUseId.
calls, turn = {}, 0  # toolUseId -> one specialist call
for message in orchestrator.messages:
    uses = [block["toolUse"] for block in message["content"] if "toolUse" in block]
    if message["role"] == "assistant" and uses:
        turn += 1
        for use in uses:
            calls[use["toolUseId"]] = {
                "turn": turn,
                "name": use["name"],
                "input": use["input"].get("input", ""),  # the text the orchestrator wrote for the specialist
            }
    for block in message["content"]:
        if "toolResult" in block and block["toolResult"]["toolUseId"] in calls:
            # A specialist's result is one content block: text, or json for a structured output.
            content = block["toolResult"]["content"][0]
            calls[block["toolResult"]["toolUseId"]]["result"] = content.get("text") or json.dumps(content.get("json"))

# Print each call with its input and result.
print(f"orchestrator turns: {orchestrator.event_loop_metrics.cycle_count}")
print("\nspecialist calls, by orchestrator turn")
for call in calls.values():
    print(f"  turn {call['turn']}  {call['name']}")
    print(f"    input : {' '.join(call['input'].split())[:200]}")
    print(f"    result: {' '.join(call['result'].split())[:200]}")

# The MCP calls behind each specialist, from its own messages.
# An Agent used as a tool starts each call from its initial messages, so these are from its latest call.
specialists = [receipt_agent, policy_agent, audit_agent]
mcp_calls = {
    agent.name: [block["toolUse"] for message in agent.messages for block in message["content"] if "toolUse" in block]
    for agent in specialists
}
print("\nMCP tool calls through the Gateway")
for name, uses in mcp_calls.items():
    for use in uses:
        print(f"  {name}: {use['name']}({json.dumps(use['input'])})")

# Tokens used per agent: accumulated_usage sums every model call the Agent has made.
print("\ntokens")
print(f"  expense_coordinator {orchestrator.event_loop_metrics.accumulated_usage['totalTokens']:>6}")
for agent in specialists:
    print(f"  {agent.name:<19} {agent.event_loop_metrics.accumulated_usage['totalTokens']:>6}")

# Check the route: receipt first, then policy and audit.
first_turn = {}  # specialist name -> the turn of its first call
for call in calls.values():
    first_turn.setdefault(call["name"], call["turn"])
assert first_turn.keys() == {"receipt_agent", "policy_agent", "audit_agent"}, f"called: {sorted(first_turn)}"
assert first_turn["receipt_agent"] < min(first_turn["policy_agent"], first_turn["audit_agent"]), (
    "policy_agent or audit_agent ran before the orchestrator had the receipt facts; read their inputs above"
)
# Each specialist called its Gateway tool, and only the tool its filter allowed.
for agent in specialists:
    assert mcp_calls[agent.name], f"{agent.name} made no MCP tool call"
    assert {use["name"] for use in mcp_calls[agent.name]} <= set(agent.tool_names), f"{agent.name} left its filter"

## A fact lost between specialists

A specialist returns its final text, and only that text, to the orchestrator. Everything else it saw, including the full tool result, stays in its private conversation. When a specialist writes for a person, a fact that another specialist needs can be dropped at that boundary.

In the next cell the receipt specialist writes one short sentence an employee can read. The request asks whether expense `expense-1042` was already claimed, and the audit specialist can check that only by employee ID.

Look for:

- Does the receipt specialist's reply include the employee ID?
- Does the orchestrator call the audit specialist, and with the employee ID?


In [ ]:
# Specialists that reply in free text for a person.
receipt_agent = Agent(
    name="receipt_agent",
    description="Looks up the receipt of one expense by expense ID and reports what is on it.",
    model=nova,
    system_prompt=(
        "You are an expense receipt specialist. Look up the receipt with your tool. Reply in one short sentence "
        "an employee can read, with the merchant, the date, and the total."
    ),
    tools=[receipt_tools],
    callback_handler=None,
)

audit_agent = Agent(
    name="audit_agent",
    description=(
        "Looks up an employee's prior expense claims by employee ID and, given the expense facts in its input, "
        "says whether the claim appears to duplicate a prior claim."
    ),
    model=nova,
    system_prompt=(
        "You are an expense audit specialist. Look up the employee's prior claims with your tool, compare them "
        "with the facts in the request, and say whether the claim appears to duplicate one of them. If a fact you "
        "need is missing from the request, say which one. Answer in at most four sentences."
    ),
    tools=[audit_tools],
    callback_handler=None,
)

# The orchestrator, with only the receipt and audit specialists.
orchestrator = Agent(
    name="expense_coordinator",
    model=sonnet,
    system_prompt=COORDINATOR_PROMPT,
    tools=[receipt_agent, audit_agent],
    callback_handler=None,
)

# Run and stream each agent into its pane.
# The orchestrator starts with an empty history, so this run does not see the previous one.
panes = LivePanes()
async for event in orchestrator.stream_async("Has the receipt for expense-1042 already been claimed?"):
    panes.show(event)

### Inspect the handoff

The employee on receipt `expense-1042` is `emp-4412`. The cell checks where that ID appears: in the receipt tool result the specialist read, in the reply it returned, and in what the audit specialist received.

In [ ]:
# Where the employee ID appears on each side of the handoff.
EMPLOYEE_ID = "emp-4412"  # the employee on receipt expense-1042 in the Lambda function's data
# receipt_agent's user messages hold its input and the MCP tool results it received.
receipt_seen = json.dumps([m["content"] for m in receipt_agent.messages if m["role"] == "user"])
# Every specialist call the orchestrator made, and which specialist each toolUseId went to.
uses = [block["toolUse"] for message in orchestrator.messages for block in message["content"] if "toolUse" in block]
names = {use["toolUseId"]: use["name"] for use in uses}
# What receipt_agent sent back to the orchestrator, as tool results.
receipt_replies = [
    json.dumps(block["toolResult"]["content"])
    for message in orchestrator.messages
    for block in message["content"]
    if "toolResult" in block and names.get(block["toolResult"]["toolUseId"]) == "receipt_agent"
]
# What the orchestrator wrote for audit_agent, and the arguments audit_agent sent to its Gateway tool.
audit_inputs = [use["input"].get("input", "") for use in uses if use["name"] == "audit_agent"]
audit_mcp = [
    block["toolUse"]["input"] for message in audit_agent.messages for block in message["content"] if "toolUse" in block
]

# Print where the ID appears.
print(f"specialist calls                         : {[use['name'] for use in uses]}")
print(f"receipt tool result has the employee ID  : {EMPLOYEE_ID in receipt_seen}")
print(
    f"receipt_agent replies with the ID        : {sum(EMPLOYEE_ID in text for text in receipt_replies)} of {len(receipt_replies)}"
)
print(
    f"audit_agent inputs with the ID           : {sum(EMPLOYEE_ID in text for text in audit_inputs)} of {len(audit_inputs)}"
)
print(f"audit_agent MCP calls                    : {audit_mcp}")

# Check that the receipt lookup ran.
assert EMPLOYEE_ID in receipt_seen, "receipt_agent did not look up the receipt"

The receipt specialist saw the employee ID in its tool result and left it out of its sentence, as its prompt asked. What happened next was the orchestrator's choice. It may have called the audit specialist without the ID, called the receipt specialist a second time, or answered that it could not tell. Each outcome costs an extra call or a weaker answer.

A result contract fixes this at the boundary. The specialist returns a small structured result instead of free text: a `status`, its `findings`, the `evidence` the findings rest on, and its `uncertainties`.

**Strands structured output:** the next cell sets `structured_output_model=SpecialistReport` on the receipt Agent. Strands asks the model for that Pydantic shape, validates it, and returns the validated fields to the orchestrator as JSON.

`evidence` and `uncertainties` are both `list[str]`, and each evidence item is one `"field: value"` pair. The field descriptions are part of the contract because the specialist model reads them. The `evidence` description asks for every field of every tool result, not only the fields the findings rest on, so the specialist keeps the employee ID that the orchestrator needs next. The rest of the cell gives the same brief receipt specialist the contract and reruns the request.

In [ ]:
# The result contract a specialist returns.
class SpecialistReport(BaseModel):
    """The result a specialist returns to the orchestrator."""

    status: Literal["complete", "partial", "failed"] = Field(description="Whether the lookup answered the request.")
    findings: list[str] = Field(description="The specialist's conclusions, one per item.")
    evidence: list[str] = Field(
        description=(
            "Every field of every tool result, as 'field: value', copied exactly. "
            "Include fields the request did not ask about."
        )
    )
    uncertainties: list[str] = Field(description="Anything missing, ambiguous, or not checked. Empty if none.")


# The receipt specialist now returns the contract.
receipt_agent = Agent(
    name="receipt_agent",
    description="Looks up the receipt of one expense by expense ID and reports what is on it.",
    model=nova,
    system_prompt=(
        "You are an expense receipt specialist. Look up the receipt with your tool. Reply in one short sentence "
        "an employee can read, with the merchant, the date, and the total."
    ),
    tools=[receipt_tools],
    structured_output_model=SpecialistReport,  # validated at the end; as a tool, returned as a json block
    callback_handler=None,
)

# The audit specialist, unchanged.
audit_agent = Agent(
    name="audit_agent",
    description=(
        "Looks up an employee's prior expense claims by employee ID and, given the expense facts in its input, "
        "says whether the claim appears to duplicate a prior claim."
    ),
    model=nova,
    system_prompt=(
        "You are an expense audit specialist. Look up the employee's prior claims with your tool, compare them "
        "with the facts in the request, and say whether the claim appears to duplicate one of them. If a fact you "
        "need is missing from the request, say which one. Answer in at most four sentences."
    ),
    tools=[audit_tools],
    callback_handler=None,
)

# The orchestrator, unchanged.
orchestrator = Agent(
    name="expense_coordinator",
    model=sonnet,
    system_prompt=COORDINATOR_PROMPT,
    tools=[receipt_agent, audit_agent],
    callback_handler=None,
)

# Run and stream each agent into its pane.
panes = LivePanes()
async for event in orchestrator.stream_async("Has the receipt for expense-1042 already been claimed?"):
    panes.show(event)

### Verify

The receipt specialist's result is now JSON with the four contract fields, and its evidence carries the employee ID even though the request never asked for it. The orchestrator can pass it on, and the audit specialist's MCP call looks up that employee.

A contract says what crosses the boundary, not how the specialist gets it. The same four fields work for the policy and audit specialists, and `uncertainties` gives the orchestrator a place to read what a specialist could not check, instead of guessing it from prose.

In [ ]:
# Collect the receipt results and the audit inputs from the run.
# The same extraction as before; this time keep receipt_agent's whole result block.
uses = [block["toolUse"] for message in orchestrator.messages for block in message["content"] if "toolUse" in block]
names = {use["toolUseId"]: use["name"] for use in uses}
receipt_results = [
    block["toolResult"]["content"][0]
    for message in orchestrator.messages
    for block in message["content"]
    if "toolResult" in block and names.get(block["toolResult"]["toolUseId"]) == "receipt_agent"
]
audit_inputs = [use["input"].get("input", "") for use in uses if use["name"] == "audit_agent"]
audit_mcp = [
    block["toolUse"]["input"] for message in audit_agent.messages for block in message["content"] if "toolUse" in block
]

# Print the contract and where the employee ID went.
# A structured output reaches the orchestrator as {"json": {...}}, the model's fields as a dict.
report = receipt_results[0].get("json", {}) if receipt_results else {}
print(json.dumps(report, indent=2))
print(f"specialist calls                       : {[use['name'] for use in uses]}")
print(
    f"audit_agent inputs with the employee ID: {sum(EMPLOYEE_ID in text for text in audit_inputs)} of {len(audit_inputs)}"
)
print(f"audit_agent MCP calls                  : {audit_mcp}")

# Check the contract fields and the audit lookup.
assert report.keys() == {"status", "findings", "evidence", "uncertainties"}, "receipt_agent did not return the contract"
assert any(EMPLOYEE_ID in item for item in report["evidence"]), "the evidence dropped the employee ID"
# audit_agent's tool arguments show whether the ID survived the handoff.
assert any(call.get("employee_id") == EMPLOYEE_ID for call in audit_mcp), (
    "audit_agent did not look up the employee; read the audit inputs above"
)

## Clean up

If you enabled observability for Lab 6, the next cell keeps the Lab 1 stack so Lab 6 can inspect its Gateway activity. Lab 6 removes the workshop-owned telemetry configuration and deletes a self-paced fallback stack in its final cleanup cell.

Otherwise, the cell deletes the stack only if this notebook deployed it. Workshop Studio event stacks are always left in place.

In [ ]:
observability_state_path = Path("..") / ".lab6-observability-state.json"
if observability_state_path.exists():
    print("Keeping the Lab 1 Gateway and its telemetry for optional Lab 6.")
else:
    cleanup("agents-as-tools")

In [ ]:
write_validation_artifact("agents-as-tools", {
    "specialists": sorted(first_turn),
    "mcp_tools": {name: [use["name"] for use in uses] for name, uses in mcp_calls.items()},
    "result_contract_fields": sorted(report),
})

## Takeaways

In Agents as Tools, the orchestrator model owns the route. It picks the specialists, the order, and the input each one receives, so the same code takes a different path for a different request. Specialists called in the same turn run concurrently, and a dependency between them forces separate turns.

The input string is the whole handoff in one direction and the specialist's result is the whole handoff in the other. A result contract with status, findings, evidence, and uncertainties keeps a specialist from dropping the facts the next one needs.

MCP tool filters keep each specialist's tool list to its own job. They run in the client, so they are not access control. In production, give each specialist its own identity and attach a Cedar policy engine to the Gateway through AgentCore Policy.

Amazon Bedrock charged for the model calls. Every specialist call is its own model call plus one more orchestrator turn to read the result, so a specialist should earn its place.

## Where to go next

The [Agents as Tools guide](https://strandsagents.com/docs/user-guide/sdk/multi-agent/agents-as-tools/) covers the `as_tool()` options, including `preserve_context`, which keeps a specialist's conversation between calls, and `delegate=True`, which returns a specialist's answer as the final response. The [Strands MCP tools guide](https://strandsagents.com/docs/user-guide/sdk/tools/mcp-tools/) covers `MCPClient` options, and the [AgentCore Gateway guide](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/gateway.html) covers other target types and policy engines.